# 53 — Chemprop Transfer from Tox21 NR Classification → PXR Regression

Tox21 nuclear receptor pathway labels (NR-AhR, NR-AR, NR-AR-LBD, NR-ER, NR-ER-LBD,
NR-PPARγ) provide binary active/inactive information for structurally-related NR targets.
The hypothesis is that a shared encoder pretrained on NR pathway classification will
produce richer compound representations for PXR, especially for activity cliff compounds
where NR-pathway membership is structurally informative.

Pipeline:
1. Load Tox21 NR data (DeepChem or parquet cache)
2. Pretrain Chemprop multi-task classifier on 6 NR binary labels
3. Fine-tune on PXR pEC50 regression (freeze → unfreeze)
4. Compare OOF RAE to nb35 (from scratch) and nb52 (ChEMBL NR pretrain)

In [ ]:
import sys, os, warnings, time
os.environ["PYTHONIOENCODING"] = "utf-8"
try:
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except AttributeError:
        pass
except Exception:
    pass
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
from pathlib import Path
import matplotlib.pyplot as plt

import chemprop
from chemprop import data as cdata, models as cmodels, nn as cnn
from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, AllChem
from rdkit import DataStructs

from pxr import data as D
from pxr.chem import to_inchikey, bemis_murcko
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS

SEED   = 42
N_FOLDS = 5
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'

# Architecture (match nb35)
DEPTH      = 3
HIDDEN_DIM = 300
FFN_LAYERS = 2
DROPOUT    = 0.1
BATCH_SIZE = 64

# Tox21 NR tasks
TOX21_NR_TASKS = ['NR-AhR', 'NR-AR', 'NR-AR-LBD', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma']
N_TOX21_TASKS  = len(TOX21_NR_TASKS)

# PXR fine-tune tasks (matching nb35)
FINETUNE_TASK_NAMES = ['pEC50', 'emax', 'pEC50_null', 'logP', 'TPSA', 'pxr_sim_max']
N_FINETUNE_TASKS    = len(FINETUNE_TASK_NAMES)
TASK_PEC50 = 0

print(f'torch {torch.__version__} | chemprop {chemprop.__version__} | lightning {L.__version__}')
print(f'device: {DEVICE}')

## 1. Load Tox21 NR data

In [ ]:
TOX21_CACHE = DATA_PROCESSED / 'tox21_nr_data.parquet'
TOX21_ENCODER_PATH = DATA_PROCESSED / 'chemprop_tox21_encoder.pt'
HAVE_TOX21 = False
tox21_df   = None

# ── Option A: load from nb37 parquet ──────────────────────────────────────
if TOX21_CACHE.exists():
    try:
        tox21_df = pd.read_parquet(TOX21_CACHE)
        print(f'Loaded Tox21 NR data from cache: {len(tox21_df):,} rows')
        print(tox21_df.columns.tolist())
        HAVE_TOX21 = True
    except Exception as e:
        print(f'Failed to load cache: {e}')

# ── Option B: DeepChem ────────────────────────────────────────────────────
if not HAVE_TOX21:
    try:
        import deepchem as dc
        print('Loading Tox21 via DeepChem...')
        tox21_tasks, tox21_datasets, _ = dc.molnet.load_tox21(
            featurizer='ECFP', splitter='scaffold'
        )
        # Extract NR-pathway tasks only
        nr_task_indices = [
            i for i, t in enumerate(tox21_tasks) if t in TOX21_NR_TASKS
        ]
        rows = []
        for split_ds in tox21_datasets:  # (train, valid, test)
            for i, smi in enumerate(split_ds.ids):
                labels = split_ds.y[i]
                for j, ti in enumerate(nr_task_indices):
                    lbl = labels[ti]
                    if not np.isnan(lbl):
                        rows.append({'smiles': smi, 'task_name': tox21_tasks[ti],
                                     'binary_label': int(lbl)})
        tox21_df = pd.DataFrame(rows)
        tox21_df.to_parquet(TOX21_CACHE, index=False)
        print(f'Loaded {len(tox21_df):,} NR-pathway labels from DeepChem. Saved to cache.')
        HAVE_TOX21 = True
    except ImportError:
        print('DeepChem not installed (pip install deepchem). Trying ChEMBL fallback...')
    except Exception as e:
        print(f'DeepChem failed: {e}. Trying ChEMBL fallback...')

# ── Option C: ChEMBL binary fallback from chembl_nr_targets.parquet ───────
if not HAVE_TOX21:
    chembl_path = DATA_EXTERNAL / 'chembl_nr_targets.parquet'
    if chembl_path.exists():
        try:
            chembl_df = pd.read_parquet(chembl_path)
            chembl_df.columns = [c.lower() for c in chembl_df.columns]
            for alt in ['pec50_mean', 'value', 'standard_value']:
                if alt in chembl_df.columns and 'pec50' not in chembl_df.columns:
                    chembl_df = chembl_df.rename(columns={alt: 'pec50'})
            if 'pec50' in chembl_df.columns and 'smiles' in chembl_df.columns:
                # Binary: active = pEC50 >= 6, inactive = pEC50 < 4.5
                active   = chembl_df[chembl_df['pec50'] >= 6.0].copy()
                inactive = chembl_df[chembl_df['pec50'] < 4.5].copy()
                active['binary_label']   = 1
                inactive['binary_label'] = 0
                target_col = 'target' if 'target' in chembl_df.columns else 'task_name'
                combined = pd.concat([active, inactive])
                if 'target' in combined.columns:
                    combined = combined.rename(columns={'target': 'task_name'})
                elif 'task_name' not in combined.columns:
                    combined['task_name'] = 'NR-AR'
                tox21_df = combined[['smiles', 'task_name', 'binary_label']].dropna()
                print(f'Using ChEMBL binary fallback: {len(tox21_df):,} rows')
                print(tox21_df['task_name'].value_counts())
                HAVE_TOX21 = True
        except Exception as e:
            print(f'ChEMBL fallback failed: {e}')

if not HAVE_TOX21:
    print('WARNING: No Tox21/ChEMBL data found.')
    print('  - Run nb37_external_data_fetch.ipynb to download ChEMBL NR data')
    print('  - Or install deepchem: pip install deepchem')
    print('  Proceeding with fine-tune from scratch (equivalent to nb35).')

if HAVE_TOX21:
    print(f'\nTox21 NR data summary:')
    print(f'  Total rows: {len(tox21_df):,}')
    print(f'  Tasks: {tox21_df["task_name"].unique().tolist()}')
    print(f'  Active fraction: {tox21_df["binary_label"].mean():.3f}')

## 2. Load PXR data + auxiliary labels

In [ ]:
tr = D.load_train()
te = D.load_test()
ct = D.load_counter()

tr_ik = tr.assign(inchikey=tr.smiles.map(to_inchikey))
ct_null = (
    ct.assign(inchikey=ct.smiles.map(to_inchikey))
    [['inchikey', 'pec50']]
    .drop_duplicates('inchikey')
    .rename(columns={'pec50': 'pec50_null'})
)
mt = tr_ik.merge(ct_null, on='inchikey', how='left')

def compute_physchem(slist):
    rows = []
    for s in slist:
        try:
            mol = Chem.MolFromSmiles(s)
            rows.append((Crippen.MolLogP(mol), Descriptors.TPSA(mol)) if mol else (np.nan, np.nan))
        except Exception:
            rows.append((np.nan, np.nan))
    return np.array(rows, dtype=np.float32)

PXR_LIGAND_SMILES = [
    'CC(C)c1ccc(cc1)S(=O)(=O)N',
    'O=C1c2ccccc2C(=O)c2ccccc21',
    'CC1(C)OC(=O)c2cc(ccc21)NC(=O)c3ccc(F)cc3',
    'Cc1ccc(cc1)S(=O)(=O)Nc2ccc(cc2)C(F)(F)F',
    'O=C(NCCC1CCCCC1)c2ccc3cc(ccc3c2)OCC(F)(F)F',
    'CCCCCCCCCCCCCC(=O)OCC(CO)OC(=O)CCCCCCCCCCCCC',
]

def compute_pxr_sim(slist):
    gen = AllChem.GetMorganGenerator(radius=2, fpSize=2048)
    ref_fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in PXR_LIGAND_SMILES
               if Chem.MolFromSmiles(s)]
    out = []
    for s in slist:
        try:
            mol = Chem.MolFromSmiles(s)
            if mol and ref_fps:
                fp = gen.GetFingerprint(mol)
                out.append(float(max(DataStructs.BulkTanimotoSimilarity(fp, ref_fps))))
            else:
                out.append(np.nan)
        except Exception:
            out.append(np.nan)
    return np.array(out, dtype=np.float32)

physchem_tr = compute_physchem(mt.smiles.tolist())
sim_max_tr  = compute_pxr_sim(mt.smiles.tolist())
physchem_te = compute_physchem(te.smiles.tolist())
sim_max_te  = compute_pxr_sim(te.smiles.tolist())

y_raw_tr = np.column_stack([
    mt['pec50'].values.astype(float),
    mt['emax'].values.astype(float),
    mt['pec50_null'].values.astype(float),
    physchem_tr[:, 0],
    physchem_tr[:, 1],
    sim_max_tr,
]).astype(np.float64)

smiles_arr = np.array(mt.smiles.tolist())
lo_clip = float(np.nanmin(y_raw_tr[:, TASK_PEC50])) - 0.5
hi_clip = float(np.nanmax(y_raw_tr[:, TASK_PEC50])) + 0.5
print(f'Train: {len(mt):,}  Test: {len(te):,}  Clip: [{lo_clip:.2f}, {hi_clip:.2f}]')

## 3. Helper functions

In [ ]:
def make_dataset_reg(smiles, y_scaled):
    dpts = [cdata.MoleculeDatapoint.from_smi(s, y=yi) for s, yi in zip(smiles, y_scaled)]
    return cdata.MoleculeDataset(dpts)


def make_dataset_cls(smiles, y_binary):
    """Binary classification dataset — uses RegressionFFN with sigmoid output."""
    dpts = [cdata.MoleculeDatapoint.from_smi(s, y=yi) for s, yi in zip(smiles, y_binary)]
    return cdata.MoleculeDataset(dpts)


def make_regression_mpnn(n_tasks):
    return cmodels.MPNN(
        message_passing=cnn.BondMessagePassing(depth=DEPTH, d_h=HIDDEN_DIM),
        agg=cnn.MeanAggregation(),
        predictor=cnn.RegressionFFN(
            n_tasks=n_tasks, n_layers=FFN_LAYERS, dropout=DROPOUT
        ),
    )


def make_loader(dataset, batch_size=BATCH_SIZE, shuffle=True):
    return cdata.build_dataloader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0)


def train_model(mpnn, loader_tr, loader_va, max_epochs, patience=10):
    callbacks = []
    if loader_va is not None:
        callbacks.append(EarlyStopping(monitor='val_loss', patience=patience, mode='min'))
    trainer = L.Trainer(
        max_epochs=max_epochs, callbacks=callbacks,
        accelerator=DEVICE, enable_progress_bar=True,
        enable_model_summary=False, logger=False,
    )
    if loader_va is not None:
        trainer.fit(mpnn, loader_tr, loader_va)
    else:
        trainer.fit(mpnn, loader_tr)
    return trainer


n_reg_params = sum(p.numel() for p in make_regression_mpnn(N_FINETUNE_TASKS).parameters())
print(f'Regression MPNN params ({N_FINETUNE_TASKS} tasks): {n_reg_params:,}')

## 4. Pretrain on Tox21 NR classification (30 epochs)

In [ ]:
if HAVE_TOX21 and not TOX21_ENCODER_PATH.exists():
    print('Building Tox21 multi-task target matrix...')
    task_to_col = {t.upper(): i for i, t in enumerate(TOX21_NR_TASKS)}
    # Also handle minor naming variations
    task_to_col['NR-PPARG']       = task_to_col.get('NR-PPAR-GAMMA', 5)
    task_to_col['NR-PPAR-GAMMA']  = 5
    task_to_col['NR-PPARGAMMA']   = 5

    tox21_smiles = tox21_df['smiles'].tolist()
    unique_smiles = list(dict.fromkeys(tox21_smiles))
    smi_to_idx = {s: i for i, s in enumerate(unique_smiles)}

    y_tox = np.full((len(unique_smiles), N_TOX21_TASKS), np.nan)
    for _, row in tox21_df.iterrows():
        col = task_to_col.get(row['task_name'].upper())
        sidx = smi_to_idx.get(row['smiles'])
        if col is not None and sidx is not None:
            y_tox[sidx, col] = float(row['binary_label'])

    # Scale binary targets to roughly zero-mean unit-variance for regression loss
    t_means_tox = np.nanmean(y_tox, axis=0)
    t_stds_tox  = np.nanstd(y_tox, axis=0, ddof=1)
    t_stds_tox  = np.where(t_stds_tox < 1e-6, 1.0, t_stds_tox)
    y_tox_sc    = (y_tox - t_means_tox) / t_stds_tox

    ds_tox    = make_dataset_cls(unique_smiles, y_tox_sc)
    loader_tox = make_loader(ds_tox, batch_size=128, shuffle=True)

    print(f'Pretraining on {len(unique_smiles):,} Tox21 compounds for 30 epochs...')
    mpnn_tox = make_regression_mpnn(N_TOX21_TASKS)
    t0 = time.time()
    train_model(mpnn_tox, loader_tox, None, max_epochs=30)
    print(f'  Pretraining done in {(time.time()-t0)/60:.1f} min')

    torch.save(mpnn_tox.message_passing.state_dict(), TOX21_ENCODER_PATH)
    print(f'  Saved Tox21 encoder: {TOX21_ENCODER_PATH}')

elif TOX21_ENCODER_PATH.exists():
    print(f'Using cached Tox21 encoder: {TOX21_ENCODER_PATH}')

else:
    print('No Tox21 data and no cached encoder — fine-tuning from scratch.')

HAVE_TOX21_ENCODER = TOX21_ENCODER_PATH.exists()

## 5. Scaffold 5-fold CV — fine-tune on PXR

In [ ]:
scaffolds = mt.smiles.map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

# Load cliff labels if available (for cliff-specific RAE analysis)
cliff_path = DATA_PROCESSED / 'cliff_labels.parquet'
if cliff_path.exists():
    cliff_df   = pd.read_parquet(cliff_path)
    if 'smiles' in cliff_df.columns:
        mt2 = mt.merge(cliff_df[['smiles', 'cliff_role']], on='smiles', how='left')
        cliff_mask = mt2['cliff_role'].fillna(0).astype(int).values != 0
    else:
        cliff_mask = np.zeros(len(mt), dtype=bool)
else:
    cliff_mask = np.zeros(len(mt), dtype=bool)

FREEZE_EPOCHS   = 10
UNFREEZE_EPOCHS = 40
PATIENCE        = 10

oof_tox21  = np.full(len(mt), np.nan)
oof_scratch = np.full(len(mt), np.nan)
fold_metrics_tox21   = []
fold_metrics_scratch = []
t_total = time.time()

for fold, (tr_idx, va_idx) in enumerate(splits):
    t_fold = time.time()
    print(f"\n{'='*60}")
    print(f'FOLD {fold+1}/{N_FOLDS}  —  {len(tr_idx):,} train / {len(va_idx):,} val')
    print(f"{'='*60}")

    y_tr_fold = y_raw_tr[tr_idx]
    t_means   = np.nanmean(y_tr_fold, axis=0)
    t_stds    = np.nanstd(y_tr_fold, axis=0, ddof=1)
    t_stds    = np.where(t_stds < 1e-6, 1.0, t_stds)

    y_tr_sc = (y_tr_fold - t_means) / t_stds
    y_va_sc = (y_raw_tr[va_idx] - t_means) / t_stds

    ds_tr = make_dataset_reg(smiles_arr[tr_idx].tolist(), y_tr_sc)
    ds_va = make_dataset_reg(smiles_arr[va_idx].tolist(), y_va_sc)
    loader_tr = make_loader(ds_tr, shuffle=True)
    loader_va = make_loader(ds_va, shuffle=False)

    # ── Tox21-pretrained + fine-tuned ─────────────────────────────────────
    mpnn_ft = make_regression_mpnn(N_FINETUNE_TASKS)
    if HAVE_TOX21_ENCODER:
        try:
            state = torch.load(TOX21_ENCODER_PATH, map_location='cpu')
            mpnn_ft.message_passing.load_state_dict(state, strict=False)
            print('  Loaded Tox21 encoder.')
        except Exception as e:
            print(f'  Encoder load failed: {e}')

    # Phase 1: freeze encoder
    for p in mpnn_ft.message_passing.parameters():
        p.requires_grad = False
    train_model(mpnn_ft, loader_tr, loader_va, max_epochs=FREEZE_EPOCHS, patience=5)

    # Phase 2: unfreeze
    for p in mpnn_ft.message_passing.parameters():
        p.requires_grad = True
    trainer_ft = train_model(mpnn_ft, loader_tr, loader_va, max_epochs=UNFREEZE_EPOCHS, patience=PATIENCE)

    raw_va = trainer_ft.predict(mpnn_ft, loader_va)
    p_sc   = torch.cat(raw_va).numpy()
    p_pxr  = np.clip(
        p_sc[:, TASK_PEC50] * t_stds[TASK_PEC50] + t_means[TASK_PEC50],
        lo_clip, hi_clip
    )
    oof_tox21[va_idx] = p_pxr
    y_true = y_raw_tr[va_idx, TASK_PEC50]
    m_ft   = compute_metrics(y_true, p_pxr)
    m_ft['fold'] = fold
    fold_metrics_tox21.append(m_ft)

    # ── From-scratch baseline ─────────────────────────────────────────────
    mpnn_sc = make_regression_mpnn(N_FINETUNE_TASKS)
    trainer_sc = train_model(
        mpnn_sc, loader_tr, loader_va,
        max_epochs=FREEZE_EPOCHS + UNFREEZE_EPOCHS, patience=PATIENCE
    )
    raw_sc = trainer_sc.predict(mpnn_sc, loader_va)
    p_sc2  = torch.cat(raw_sc).numpy()
    p_pxr_sc = np.clip(
        p_sc2[:, TASK_PEC50] * t_stds[TASK_PEC50] + t_means[TASK_PEC50],
        lo_clip, hi_clip
    )
    oof_scratch[va_idx] = p_pxr_sc
    m_sc = compute_metrics(y_true, p_pxr_sc)
    m_sc['fold'] = fold
    fold_metrics_scratch.append(m_sc)

    elapsed = time.time() - t_fold
    print(f'  Tox21+FT: RAE={m_ft["RAE"]:.4f}  |  Scratch: RAE={m_sc["RAE"]:.4f}  ({elapsed/60:.1f} min)')

print(f'\nTotal CV time: {(time.time()-t_total)/60:.1f} min')

## 6. Analysis: Tox21 transfer vs from-scratch — overall and cliff-specific RAE

In [ ]:
y_all = y_raw_tr[:, TASK_PEC50]

oof_rae_tox21   = rae_fn(y_all, oof_tox21)
oof_rae_scratch = rae_fn(y_all, oof_scratch)

rae_tox21_cliff   = rae_fn(y_all[cliff_mask], oof_tox21[cliff_mask])   if cliff_mask.sum() > 5 else float('nan')
rae_scratch_cliff = rae_fn(y_all[cliff_mask], oof_scratch[cliff_mask]) if cliff_mask.sum() > 5 else float('nan')

cv_tox21   = pd.DataFrame(fold_metrics_tox21)
cv_scratch = pd.DataFrame(fold_metrics_scratch)

print('=== Tox21 + Fine-tune ===')
print(cv_tox21[['fold', 'RAE', 'MAE', 'Spearman']].to_string(index=False))
print(f'OOF RAE (overall): {oof_rae_tox21:.4f}  cliff: {rae_tox21_cliff:.4f}')

print('\n=== From Scratch ===')
print(cv_scratch[['fold', 'RAE', 'MAE', 'Spearman']].to_string(index=False))
print(f'OOF RAE (overall): {oof_rae_scratch:.4f}  cliff: {rae_scratch_cliff:.4f}')

# Compare to nb35
nb35_path = DATA_PROCESSED / 'oof_chemprop_aux.npy'
if nb35_path.exists():
    oof_nb35 = np.load(nb35_path)
    # Align length (nb35 may have different row count if train was re-loaded)
    if len(oof_nb35) == len(y_all):
        rae_nb35 = rae_fn(y_all, oof_nb35)
        rae_nb35_cliff = rae_fn(y_all[cliff_mask], oof_nb35[cliff_mask]) if cliff_mask.sum() > 5 else float('nan')
        print(f'\nnb35 (scratch, 6-head): OOF RAE = {rae_nb35:.4f}  cliff = {rae_nb35_cliff:.4f}')
    else:
        rae_nb35 = float('nan')
        print('nb35 OOF length mismatch — skipping comparison.')
else:
    rae_nb35 = float('nan')
    print('oof_chemprop_aux.npy not found — run nb35.')

print(f'\nDelta Tox21+FT vs Scratch: {oof_rae_tox21 - oof_rae_scratch:+.4f}')
if not np.isnan(rae_tox21_cliff) and not np.isnan(rae_scratch_cliff):
    print(f'Delta cliff-specific:      {rae_tox21_cliff - rae_scratch_cliff:+.4f}')

best_label = 'tox21+FT' if oof_rae_tox21 <= oof_rae_scratch else 'scratch'
oof_best   = oof_tox21  if oof_rae_tox21 <= oof_rae_scratch else oof_scratch
print(f'\nBest variant: {best_label}')

In [ ]:
# ── Plots ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x = np.arange(N_FOLDS)
axes[0].bar(x - 0.2, cv_tox21['RAE'],   0.35, label='Tox21+FT', color='steelblue', alpha=0.8)
axes[0].bar(x + 0.2, cv_scratch['RAE'], 0.35, label='Scratch',  color='tomato',    alpha=0.8)
if not np.isnan(rae_nb35):
    axes[0].axhline(rae_nb35, color='black', ls='--', lw=1.2, label=f'nb35 ({rae_nb35:.4f})')
axes[0].set_xlabel('Fold')
axes[0].set_ylabel('OOF RAE')
axes[0].set_title('Tox21+FT vs Scratch — per-fold RAE')
axes[0].legend()
axes[0].set_xticks(x)

# Cliff vs non-cliff RAE comparison
categories = ['Overall', 'Cliff compounds', 'Non-cliff']
noncliff_mask = ~cliff_mask
tox21_vals = [
    oof_rae_tox21,
    rae_tox21_cliff,
    rae_fn(y_all[noncliff_mask], oof_tox21[noncliff_mask]) if noncliff_mask.sum() > 5 else float('nan'),
]
scratch_vals = [
    oof_rae_scratch,
    rae_scratch_cliff,
    rae_fn(y_all[noncliff_mask], oof_scratch[noncliff_mask]) if noncliff_mask.sum() > 5 else float('nan'),
]
xc = np.arange(3)
axes[1].bar(xc - 0.2, tox21_vals,   0.35, label='Tox21+FT', color='steelblue', alpha=0.8)
axes[1].bar(xc + 0.2, scratch_vals, 0.35, label='Scratch',  color='tomato',    alpha=0.8)
axes[1].set_xticks(xc)
axes[1].set_xticklabels(categories)
axes[1].set_ylabel('RAE')
axes[1].set_title('RAE breakdown: overall vs cliff vs non-cliff')
axes[1].legend()

plt.tight_layout()
fig_path = DATA_PROCESSED / 'figures' / 'chemprop_tox21_transfer.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=120)
plt.show()

## 7. Final model + save outputs

In [ ]:
print(f'Training final model ({best_label}) on all data...')
t_means_full = np.nanmean(y_raw_tr, axis=0)
t_stds_full  = np.nanstd(y_raw_tr, axis=0, ddof=1)
t_stds_full  = np.where(t_stds_full < 1e-6, 1.0, t_stds_full)
y_full_sc = (y_raw_tr - t_means_full) / t_stds_full

ds_full     = make_dataset_reg(smiles_arr.tolist(), y_full_sc)
loader_full = make_loader(ds_full, shuffle=True)

mpnn_final = make_regression_mpnn(N_FINETUNE_TASKS)
if best_label == 'tox21+FT' and HAVE_TOX21_ENCODER:
    try:
        state = torch.load(TOX21_ENCODER_PATH, map_location='cpu')
        mpnn_final.message_passing.load_state_dict(state, strict=False)
        print('  Loaded Tox21 encoder for final model.')
    except Exception as e:
        print(f'  Encoder load failed: {e}')

trainer_final = train_model(
    mpnn_final, loader_full, None,
    max_epochs=FREEZE_EPOCHS + UNFREEZE_EPOCHS
)

te_dpts   = [cdata.MoleculeDatapoint.from_smi(s) for s in te.smiles]
te_ds     = cdata.MoleculeDataset(te_dpts)
te_loader = make_loader(te_ds, batch_size=128, shuffle=False)

raw_te    = trainer_final.predict(mpnn_final, te_loader)
p_te_sc   = torch.cat(raw_te).numpy()
te_preds  = np.clip(
    p_te_sc[:, TASK_PEC50] * t_stds_full[TASK_PEC50] + t_means_full[TASK_PEC50],
    lo_clip, hi_clip
)
print(f'Test preds — min={te_preds.min():.2f}  median={np.median(te_preds):.2f}  max={te_preds.max():.2f}')

In [ ]:
np.save(DATA_PROCESSED / 'oof_chemprop_tox21.npy', oof_best)
np.save(DATA_PROCESSED / 'te_chemprop_tox21.npy',  te_preds)
print(f'Saved oof_chemprop_tox21.npy  (OOF RAE = {rae_fn(y_all, oof_best):.4f})')

sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'pEC50': te_preds,
})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out_path = SUBMISSIONS / '53_chemprop_tox21_transfer.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(sub['pEC50'].describe().round(3))

## Summary

| Model | OOF RAE (overall) | OOF RAE (cliff) | Notes |
|---|---|---|---|
| nb35 (from scratch) | see above | see above | 6-head auxiliary |
| Scratch (this CV) | see above | see above | same arch |
| **Tox21+FT (this)** | **see above** | **see above** | NR cls pretrain |

**Saved:**
- `data/processed/chemprop_tox21_encoder.pt`  — Tox21-pretrained encoder
- `data/processed/oof_chemprop_tox21.npy`     — best OOF pEC50
- `data/processed/te_chemprop_tox21.npy`      — test pEC50
- `submissions/53_chemprop_tox21_transfer.csv`